# 有趣的指标
最长的歌
最短的歌
前奏最长
前奏最短
歌词字数最多
歌词字数最少
歌词重复率最高
歌词重复率最低
歌名最长

In [23]:
import zlib
import re

import pandas as pd
import json

## 歌词重复率

In [20]:
def get_text_repetition_stats(text, min_len=4):
    """
    中英文兼容版：支持汉字统计与英文单词统计
    """
    # 1. 提取有效字符：保留汉字、字母和单词间的空格
    # \u4e00-\u9fa5 汉字 | a-zA-Z 英文
    # 我们先清洗掉标点符号，保留空格以区分英文单词
    clean_text_with_spaces = " ".join(re.findall(r'[\u4e00-\u9fa5]|[a-zA-Z]+', text))
    
    # 2. 计算 total_chars (英文按单词算一个单位，汉字按个算)
    # 但为了保持和之前 total_chars 逻辑一致，我们通常按“字符长度”计算
    # 移除所有多余空格进行基础统计
    clean_text = re.sub(r'\s+', '', clean_text_with_spaces)
    total_chars = len(clean_text)
    
    if total_chars == 0:
        return {"total_chars": 0, "compression_repetition_rate": "0.00%", "repeated_chars_total": 0, "repeat_detail": []}

    # 3. 压缩率 (基于编码后的字节)
    encoded = clean_text.encode('utf-8')
    compression_rate = (1 - len(zlib.compress(encoded, level=9)) / len(encoded)) * 100

    # 4. 重复标记逻辑 (使用 mask 标记被覆盖的字符位)
    covered_mask = [0] * len(clean_text)
    repeated_segments = {}

    # 遍历不同长度的子串
    for length in range(20, min_len - 1, -1):
        for i in range(len(clean_text) - length + 1):
            sub = clean_text[i:i + length]
            # 只有出现 2 次及以上，且不是已被记录的长子串
            if clean_text.count(sub) > 1:
                if not any(sub in existing for existing in repeated_segments):
                    repeated_segments[sub] = clean_text.count(sub)
                
                # 标记位置
                for match in re.finditer(f'(?={re.escape(sub)})', clean_text):
                    start = match.start()
                    for k in range(start, start + length):
                        covered_mask[k] = 1

    rep_chars_total = sum(covered_mask)

    return {
        "total_chars": total_chars,
        "compression_repetition_rate": f"{compression_rate:.2f}%",
        "repeated_chars_total": rep_chars_total,
        # "repeat_detail": sorted(repeated_segments.items(), key=lambda x: x[1], reverse=True)
    }

## 前奏时长

In [48]:
# start_time为分秒格式，如00:15.71，取.前部分，并转换为秒，作为前奏时长
def parse_intro_duration(start_time_series):
    """
    将start_time(格式如00:15.71)转换为前奏时长(秒)
    
    参数:
        start_time_series: pandas Series, 包含start_time字符串
    
    返回:
        pandas Series, 前奏时长(秒)
    """
    start_time_parts = start_time_series.astype(str).str.split('.', n=1, expand=True)[0]
    start_time_split = start_time_parts.str.split(':', n=1, expand=True)
    
    minutes = pd.to_numeric(start_time_split[0], errors='coerce').fillna(0).astype(int)
    seconds = pd.to_numeric(start_time_split[1], errors='coerce').fillna(0).astype(int)
    
    return minutes * 60 + seconds

## 歌曲时长

In [52]:
# 歌曲时长，将duration转为xx分xx秒文本格式，并保留duration字段
def format_duration(duration_seconds):
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes}分{seconds}秒"

## 歌名字数

In [116]:
def count_song_name(name):
    if not name:
        return 0
    # 1. 匹配所有英文单词 (Now, You, See, Me)
    english_words = re.findall(r'[a-zA-Z0-9]+', name)
    # 2. 匹配所有中文字符 (去除空格后的非英文字符)
    # 先去掉空格，再去掉英文和数字，剩下的就是中文字符或符号
    remaining = re.sub(r'\s+|[a-zA-Z0-9]+', '', name)
    
    # 单词数 + 中文字符数
    return len(english_words) + len(remaining)

## 计算

In [125]:
def analyze_songs_metrics(path_prefix):
    songs_df = pd.read_csv(path_prefix + "cleared_song_data.csv")
    lyrics_df = pd.read_json(path_prefix + "cleared_lyric_data.json",
                             convert_dates=False)
    # 删除lyrics_df中没有歌词的行
    lyrics_df = lyrics_df[lyrics_df['has_lyric'] == 1]
    # lyrics_df 连接 songs_df， 以song_id为键，以lyrics_df为主，保留所有歌词数据，重复列只保留lyrics_df的
    df = pd.merge(lyrics_df,
                  songs_df,
                  on='song_id',
                  how='left',
                  suffixes=('', '_song'))
    df['intro_duration_seconds'] = parse_intro_duration(df['start_time'])
    # 歌曲时长
    df['duration_text'] = df['duration'].apply(format_duration)
    # 歌词字数，重复率等指标
    metrics = df['lyrics_text'].apply(get_text_repetition_stats)
    metrics_df = pd.DataFrame(metrics.tolist())
    df = pd.concat([df, metrics_df], axis=1)
    # 歌名清洗，去掉括号及其中内容，去掉引号，仅计算汉字数和单词数
    df['clean_song_name'] = (df['song_name'].fillna('').str.replace(
        r'（[^）]*）', '',
        regex=True).str.replace(r'【[^】]*】', '', regex=True).str.replace(
            r'\([^)]*\)', '', regex=True).str.replace(r'[\'"“”]',
                                                      '',
                                                      regex=True).str.strip())
    # 歌名长度。不含空格的字符长度，英文单词按一个单位算
    df['song_name_length'] = df['clean_song_name'].apply(count_song_name)
    return df

In [181]:
def get_extreme_metrics(df):
    # 1. 预处理：数值化重复率以便比较
    comp_numeric = pd.to_numeric(
        df['compression_repetition_rate'].str.rstrip('%'), errors='coerce')

    def _get_extrema(col_name, use_series=None, text_col=None):
        """
        内部辅助函数：获取极值对应的歌曲名拼接字符串、数值、以及对应的文本展示字段
        """
        series = use_series if use_series is not None else df[col_name]
        if series.empty or series.isna().all():
            return "无", 0, "无", 0, "0", "0"

        v_max, v_min = series.max(), series.min()

        # 提取所有并列的歌名
        names_max = "，".join(df.loc[series == v_max,
                                    'song_name'].unique().astype(str))
        names_min = "，".join(df.loc[series == v_min,
                                    'song_name'].unique().astype(str))

        # 提取对应的展示文本 (取并列组中的第一个)
        t_max = str(v_max)
        t_min = str(v_min)
        if text_col and text_col in df.columns:
            t_max = df.loc[series == v_max, text_col].iloc[0]
            t_min = df.loc[series == v_min, text_col].iloc[0]

        return names_max, v_max, names_min, v_min, t_max, t_min

    # 2. 计算各个维度的极值
    # 时长：传入 duration_text 字段用于展示
    d_names_max, d_max, d_names_min, d_min, d_text_max, d_text_min = _get_extrema(
        'duration', text_col='duration_text')

    # 前奏
    i_names_max, i_max, i_names_min, i_min, _, _ = _get_extrema(
        'intro_duration_seconds')

    # 字数
    c_names_max, c_max, c_names_min, c_min, _, _ = _get_extrema('total_chars')

    # 重复率
    r_names_max, r_max, r_names_min, r_min, _, _ = _get_extrema(
        None, use_series=comp_numeric)

    # 歌名长度
    n_names_max, n_max, _, _, _, _ = _get_extrema('song_name_length')

    # 3. 组装结果字典
    metrics = {
        # 时长统计 - 使用文本化字段
        "最长的歌": d_names_max,
        "最长的歌时长": d_text_max,
        "longest_song_number": int(d_max),
        "最短的歌": d_names_min,
        "最短的歌时长": d_text_min,
        "shortest_song_number": int(d_min),

        # 前奏统计
        "前奏最长": i_names_max,
        "前奏最长时长": f"{i_max}秒",
        "longest_intro_number": int(i_max),
        "前奏最短": i_names_min,
        "前奏最短时长": f"{i_min}秒",
        "shortest_intro_number": int(i_min),

        # 歌词统计
        "歌词字数最多": c_names_max,
        "歌词字数最多字数": f"{c_max}字",
        "most_lyrics_number": int(c_max),
        "歌词字数最少": c_names_min,
        "歌词字数最少字数": f"{c_min}字",
        "least_lyrics_number": int(c_min),

        # 重复率统计
        "歌词重复率最高": r_names_max,
        "歌词重复率最高重复率": f"{r_max}%",
        "歌词重复率最低": r_names_min,
        "歌词重复率最低重复率": f"{r_min}%",

        # 歌名统计
        "歌名最长": n_names_max,
        "歌名最长字数": f"{n_max}字",
        "longest_song_name_number": int(n_max)
    }

    metrics_vue_data = {
        "singer": df['artist_name'].iloc[0],
        "song": {
            "title":
            "歌曲时长（秒）",
            "category": [
                f"最长：{d_names_max} - {d_text_max}",
                f"最短：{d_names_min} - {d_text_min}"
            ],
            "number": [int(d_max), int(d_min)]
        },
        "intro": {
            "title":
            "前奏时长（秒）",
            "category":
            [f"最长：{i_names_max} - {i_max}秒", f"最短：{i_names_min} - {i_min}秒"],
            "number": [int(i_max), int(i_min)]
        },
        "lyrics": {
            "title":
            "歌词字数",
            "category":
            [f"最多：{c_names_max} - {c_max}字", f"最少：{c_names_min} - {c_min}字"],
            "number": [int(c_max), int(c_min)]
        },
        "repetition": {
            "title":
            "歌词重复率（%）",
            "category":
            [f"最高：{r_names_max} - {r_max}%", f"最低：{r_names_min} - {r_min}%"],
            "number": [float(r_max), float(r_min)]
        },
        # "song_name": {
        #     "category": [f"最长：{n_names_max} - {n_max}字"],
        #     "number": [int(n_max)]
        # }
    }
    return metrics, metrics_vue_data
    # return metrics_vue_data

# main

In [185]:
file_path_prefix = "data/jaychou/"

In [188]:
file_path_prefix = "data/jaychou/"
df = analyze_songs_metrics(file_path_prefix)
jay_res = get_extreme_metrics(df)
jay_res

({'最长的歌': '以父之名',
  '最长的歌时长': '5分42秒',
  'longest_song_number': 342,
  '最短的歌': '阳明山',
  '最短的歌时长': '2分32秒',
  'shortest_song_number': 152,
  '前奏最长': '半兽人',
  '前奏最长时长': '63秒',
  'longest_intro_number': 63,
  '前奏最短': '公主病，免费教学录影带',
  '前奏最短时长': '0秒',
  'shortest_intro_number': 0,
  '歌词字数最多': '以父之名',
  '歌词字数最多字数': '1226字',
  'most_lyrics_number': 1226,
  '歌词字数最少': '蒲公英的约定',
  '歌词字数最少字数': '227字',
  'least_lyrics_number': 227,
  '歌词重复率最高': '可爱女人',
  '歌词重复率最高重复率': '82.31%',
  '歌词重复率最低': '最伟大的作品',
  '歌词重复率最低重复率': '38.67%',
  '歌名最长': '给我一首歌的时间',
  '歌名最长字数': '8字',
  'longest_song_name_number': 8},
 {'singer': '周杰伦',
  'song': {'title': '歌曲时长（秒）',
   'category': ['最长：以父之名 - 5分42秒', '最短：阳明山 - 2分32秒'],
   'number': [342, 152]},
  'intro': {'title': '前奏时长（秒）',
   'category': ['最长：半兽人 - 63秒', '最短：公主病，免费教学录影带 - 0秒'],
   'number': [63, 0]},
  'lyrics': {'title': '歌词字数',
   'category': ['最多：以父之名 - 1226字', '最少：蒲公英的约定 - 227字'],
   'number': [1226, 227]},
  'repetition': {'title': '歌词重复率（%）',
   'category':

In [ ]:
df[['song_name', 'start_time']].sort_values(by='start_time', ascending=True)

,song_name,start_time
148,免费教学录影带,00:00.49
101,公主病,00:00.81
72,甜甜的,00:01.77
134,水手怕水,00:02.03
146,魔术先生,00:02.25
...,...,...
84,龙拳,00:49.21
116,龙战骑士,00:49.33
130,蓝色风暴,00:51.39
81,飘移,00:53.26


In [193]:
file_path_prefix = "data/mayday/"
df = analyze_songs_metrics(file_path_prefix)
mayday_res = get_extreme_metrics(df)
mayday_res

({'最长的歌': '温柔 (还你自由版)',
  '最长的歌时长': '7分6秒',
  'longest_song_number': 426,
  '最短的歌': '为什么（今日的爱情）',
  '最短的歌时长': '1分53秒',
  'shortest_song_number': 113,
  '前奏最长': '孙悟空',
  '前奏最长时长': '63秒',
  'longest_intro_number': 63,
  '前奏最短': '派对动物，夜访吸血鬼，雨眠',
  '前奏最短时长': '0秒',
  'shortest_intro_number': 0,
  '歌词字数最多': '干杯',
  '歌词字数最多字数': '700字',
  'most_lyrics_number': 700,
  '歌词字数最少': '后青春期的诗，金多虾',
  '歌词字数最少字数': '175字',
  'least_lyrics_number': 175,
  '歌词重复率最高': '天使',
  '歌词重复率最高重复率': '77.02%',
  '歌词重复率最低': '后青春期的诗',
  '歌词重复率最低重复率': '25.14%',
  '歌名最长': '有些事现在不做 一辈子都不会做了',
  '歌名最长字数': '15字',
  'longest_song_name_number': 15},
 {'singer': '五月天',
  'song': {'title': '歌曲时长（秒）',
   'category': ['最长：温柔 (还你自由版) - 7分6秒', '最短：为什么（今日的爱情） - 1分53秒'],
   'number': [426, 113]},
  'intro': {'title': '前奏时长（秒）',
   'category': ['最长：孙悟空 - 63秒', '最短：派对动物，夜访吸血鬼，雨眠 - 0秒'],
   'number': [63, 0]},
  'lyrics': {'title': '歌词字数',
   'category': ['最多：干杯 - 700字', '最少：后青春期的诗，金多虾 - 175字'],
   'number': [700, 175]},
  'repetition': 

In [194]:
df[['song_name', 'start_time']].sort_values(by='start_time', ascending=True)

,song_name,start_time
91,雨眠,00:00.75
27,夜访吸血鬼,00:00.80
4,派对动物,00:00.81
127,生命有一种绝对,00:01.43
45,后青春期的诗,00:01.84
...,...,...
67,雌雄同体,00:48.03
128,温柔 (还你自由版),00:50.56
16,盛夏光年,00:55.25
109,轻功,00:55.40


In [174]:
res = {"jay": jay_res, "mayday": mayday_res}
# 保存为json文件
with open('data/extreme_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=4)

In [175]:
516/3*4

688.0

In [180]:
aaa = "Ave，Maria，grazia，ricevuta，per，la，mia，famiglia。Con，risentito，con，un'amorevole，divino，amen。Grazie，chiedo，a，te，o，signore，divino。In，questo，giorno，di，grazia，prego，per，te。Ave，Maria，piena，di，grazia。Il，signore，e，con，te。Sia，fatta，la，tua，volonta。Così，in，cielo，e，così，in，terra，neil，nome。Del，padre，del，figliolo，e，dello，spirito，santo，amen。微凉的晨露沾湿黑礼服。石板路有雾父在低诉。无奈的觉悟只能更残酷。一切都为了通往圣堂的路。吹不散的雾隐没了意图。谁轻柔踱步停住。还来不及哭穿过的子弹就带走温度。我们每个人都有罪。犯着不同的罪。我能决定谁对。谁又该要沉睡。争论不能解决。在永无止境的夜。关掉你的嘴。唯一的恩惠。挡在前面的人都有罪。后悔也无路可退。以父之名判决。那感觉没有适合字汇。就像边笑边掉泪。凝视着完全的黑。阻挡悲剧蔓延的悲剧会让我沉醉。低头亲吻我的左手。换取被宽恕的承诺。老旧管风琴在角落。一直一直一直伴奏。黑色帘幕被风吹动。阳光无言地穿透。洒向那群被我驯服后的兽。沉默地喊叫沉默地喊叫。孤单开始发酵。不停对着我嘲笑。回忆逐渐延烧。曾经纯真的画面。残忍地温柔出现。脆弱时间到。我们一起来祷告。仁慈的父我已坠入。看不见罪的国度。请原谅我的自负。Ah，ya，ya，check，it，check，it，ah，ya。没人能说没人可说。好难承受。荣耀的背后刻着一道孤独。Ah，ya，ya，check，it，check，it，ah，ya。闭上双眼我又看见。当年那梦的画面。天空是濛濛的雾。Ah，ya，ya，check，it，check，it，ah，ya。父亲牵着我的双手。轻轻走过。清晨那安安静静的石板路。Ah，ya，ya，check，it，check，it，ah，ya。Pie，Jesu。Qui，tollis，peccata。Dona，eis，requiem。低头亲吻我的左手。换取被宽恕的承诺。老旧管风琴在角落。一直一直一直伴奏。黑色帘幕被风吹动。阳光无言地穿透。洒向那群被我驯服后的兽。沉默地喊叫沉默地喊叫。孤单开始发酵。不停对着我嘲笑。回忆逐渐延烧。曾经纯真的画面。残忍地温柔出现。脆弱时间到。我们一起来祷告。仁慈的父我已坠入。看不见罪的国度。请原谅我的自负。Ah，ya，ya，check，it，check，it，ah，ya。没人能说没人可说。好难承受。荣耀的背后刻着一道孤独。Ah，ya，ya，check，it，check，it，ah，ya。仁慈的父我已坠入。看不见罪的国度。请原谅我，我的自负。刻着一道孤独。仁慈的父我已坠入。看不见罪的国度。请原谅我的自负。Ah，ya，ya，check，it，check，it，ah，ya。没人能说没人可说。好难承受。荣耀的背后刻着一道孤独。Ah，ya，ya，check，it，check，it，ah，ya。闭上双眼我又看见。（斑驳的家徽擦拭了一夜）。当年那梦的画面。（孤独的光辉，才懂的感觉）。天空是濛濛的雾。（烛光，不不停的摇晃）。猫头鹰在窗棂上，对着远方眺望。父亲牵着我的双手。（通向大厅的长廊）。轻轻走过，清晨那。（一样说不出的沧桑）。安安静静的石板路。（没有喧嚣，只有宁静围绕）。我，慢慢睡着。天，刚刚破晓。"
len(aaa)

1486